# Advanced Problems with Solutions: Rational Numbers

**Kernel:** Python 3.13

This notebook contains advanced practice problems on `fractions.Fraction`, rational arithmetic, float exactness, normalization, denominator limits, parsing, and robust utility design.

In [1]:
from fractions import Fraction
from decimal import Decimal
import math

## Problem 1 — Constructor Deep Dive

Predict the result of each constructor call before running the cell.

Focus on:

- integer construction
- numerator/denominator normalization
- negative denominator handling
- string construction
- float exactness
- Decimal exactness

In [2]:
cases = [
    ("Fraction()", lambda: Fraction()),
    ("Fraction(5)", lambda: Fraction(5)),
    ("Fraction(8, 16)", lambda: Fraction(8, 16)),
    ("Fraction(1, -4)", lambda: Fraction(1, -4)),
    ("Fraction(-2, -6)", lambda: Fraction(-2, -6)),
    ("Fraction('10.5')", lambda: Fraction("10.5")),
    ("Fraction('22/7')", lambda: Fraction("22/7")),
    ("Fraction(0.3)", lambda: Fraction(0.3)),
    ("Fraction(Decimal('0.3'))", lambda: Fraction(Decimal("0.3"))),
]

for label, fn in cases:
    try:
        print(f"{label:<32} -> {fn()!r}")
    except Exception as ex:
        print(f"{label:<32} -> {type(ex).__name__}: {ex}")

Fraction()                       -> Fraction(0, 1)
Fraction(5)                      -> Fraction(5, 1)
Fraction(8, 16)                  -> Fraction(1, 2)
Fraction(1, -4)                  -> Fraction(-1, 4)
Fraction(-2, -6)                 -> Fraction(1, 3)
Fraction('10.5')                 -> Fraction(21, 2)
Fraction('22/7')                 -> Fraction(22, 7)
Fraction(0.3)                    -> Fraction(5404319552844595, 18014398509481984)
Fraction(Decimal('0.3'))         -> Fraction(3, 10)


### Solution 1

`Fraction` always stores rational numbers in normalized form:

- denominator is positive;
- numerator and denominator are reduced by their greatest common divisor;
- `Fraction(0.3)` captures the exact binary floating-point value of `0.3`;
- `Fraction(Decimal("0.3"))` captures exactly three tenths.

## Problem 2 — Inspect Fraction Invariants

Write `inspect_fraction(x)` that returns a dictionary containing:

- the fraction itself;
- numerator;
- denominator;
- whether the denominator is positive;
- whether the fraction is in lowest terms.

Then test it on several values.

In [3]:
def inspect_fraction(x: Fraction) -> dict[str, object]:
    if not isinstance(x, Fraction):
        raise TypeError("x must be a Fraction")

    return {
        "fraction": x,
        "numerator": x.numerator,
        "denominator": x.denominator,
        "denominator_is_positive": x.denominator > 0,
        "is_lowest_terms": math.gcd(x.numerator, x.denominator) == 1,
    }


examples = [
    Fraction(8, 16),
    Fraction(1, -4),
    Fraction(-50, -100),
    Fraction(0),
    Fraction(22, 7),
]

for x in examples:
    print(inspect_fraction(x))

{'fraction': Fraction(1, 2), 'numerator': 1, 'denominator': 2, 'denominator_is_positive': True, 'is_lowest_terms': True}
{'fraction': Fraction(-1, 4), 'numerator': -1, 'denominator': 4, 'denominator_is_positive': True, 'is_lowest_terms': True}
{'fraction': Fraction(1, 2), 'numerator': 1, 'denominator': 2, 'denominator_is_positive': True, 'is_lowest_terms': True}
{'fraction': Fraction(0, 1), 'numerator': 0, 'denominator': 1, 'denominator_is_positive': True, 'is_lowest_terms': True}
{'fraction': Fraction(22, 7), 'numerator': 22, 'denominator': 7, 'denominator_is_positive': True, 'is_lowest_terms': True}


### Solution 2

A `Fraction` object enforces its own canonical representation.

Even if the user constructs `Fraction(8, 16)`, Python stores it as `Fraction(1, 2)`.

## Problem 3 — Exact Float Error

Write `float_error(x, exact)` that compares a float with an exact rational target.

Example:

```python
float_error(0.3, Fraction(3, 10))
```

The function should return the exact rational error:

```python
Fraction(x) - exact
```

In [4]:
def float_error(x: float, exact: Fraction) -> Fraction:
    if not isinstance(x, float):
        raise TypeError("x must be a float")
    if not isinstance(exact, Fraction):
        raise TypeError("exact must be a Fraction")

    return Fraction(x) - exact


examples = [
    (0.1, Fraction(1, 10)),
    (0.2, Fraction(1, 5)),
    (0.3, Fraction(3, 10)),
    (0.5, Fraction(1, 2)),
    (0.125, Fraction(1, 8)),
]

for x, exact in examples:
    err = float_error(x, exact)
    print(f"{x!r:<6} exact target={exact!s:<6} error={err!s:<35} float(error)={float(err): .18e}")

0.1    exact target=1/10   error=1/180143985094819840                float(error)= 5.551115123125783010e-18
0.2    exact target=1/5    error=1/90071992547409920                 float(error)= 1.110223024625156602e-17
0.3    exact target=3/10   error=-1/90071992547409920                float(error)=-1.110223024625156602e-17
0.5    exact target=1/2    error=0                                   float(error)= 0.000000000000000000e+00
0.125  exact target=1/8    error=0                                   float(error)= 0.000000000000000000e+00


### Solution 3

Some decimal fractions, such as `0.5` and `0.125`, are exactly representable in binary floating point.

Others, such as `0.1`, `0.2`, and `0.3`, are not exactly representable.

`Fraction(x)` exposes the exact rational value stored by the float.

## Problem 4 — Avoid Float Contamination

Evaluate the following expressions and explain why some return `Fraction` while others return `float`.

```python
Fraction(1, 3) + Fraction(1, 6)
Fraction(1, 3) + 0.25
Fraction(1, 3) + Fraction(0.25)
Fraction(1, 3) + Fraction("0.25")
```

In [5]:
expressions = [
    ("Fraction(1, 3) + Fraction(1, 6)", lambda: Fraction(1, 3) + Fraction(1, 6)),
    ("Fraction(1, 3) + 0.25", lambda: Fraction(1, 3) + 0.25),
    ("Fraction(1, 3) + Fraction(0.25)", lambda: Fraction(1, 3) + Fraction(0.25)),
    ('Fraction(1, 3) + Fraction("0.25")', lambda: Fraction(1, 3) + Fraction("0.25")),
]

for label, fn in expressions:
    result = fn()
    print(f"{label:<42} -> {result!r:<30} type={type(result).__name__}")

Fraction(1, 3) + Fraction(1, 6)            -> Fraction(1, 2)                 type=Fraction
Fraction(1, 3) + 0.25                      -> 0.5833333333333333             type=float
Fraction(1, 3) + Fraction(0.25)            -> Fraction(7, 12)                type=Fraction
Fraction(1, 3) + Fraction("0.25")          -> Fraction(7, 12)                type=Fraction


### Solution 4

Mixing `Fraction` with `float` generally produces a `float`.

To keep exact rational arithmetic, convert decimal quantities using strings or `Decimal` when appropriate:

```python
Fraction("0.25")
Fraction(Decimal("0.25"))
```

Avoid writing `Fraction(0.1)` when you mean exactly one tenth.

## Problem 5 — Implement Rational Power Rules

Write `safe_fraction_power(base, exponent)`.

Rules:

- `base` must be a `Fraction`;
- if `exponent` is an `int`, return exact `Fraction` arithmetic;
- if `exponent` is not an `int`, raise `TypeError`;
- handle negative integer exponents.

In [6]:
def safe_fraction_power(base: Fraction, exponent: int) -> Fraction:
    if not isinstance(base, Fraction):
        raise TypeError("base must be a Fraction")

    if type(exponent) is not int:
        raise TypeError("exponent must be a plain int")

    return base ** exponent


tests = [
    (Fraction(2, 3), 3),
    (Fraction(2, 3), -3),
    (Fraction(-2, 3), 2),
    (Fraction(-2, 3), 3),
]

for base, exponent in tests:
    print(f"{base!s:>6} ** {exponent:>3} = {safe_fraction_power(base, exponent)!s}")

   2/3 **   3 = 8/27
   2/3 **  -3 = 27/8
  -2/3 **   2 = 4/9
  -2/3 **   3 = -8/27


### Solution 5

Integer powers of rational numbers are rational.

Negative exponents invert the base:

```python
Fraction(2, 3) ** -3 == Fraction(27, 8)
```

Non-integer powers can produce irrational results, so this function rejects them.

## Problem 6 — Parse User Ratios Safely

Write `parse_ratio(text)` that accepts strings such as:

```python
"3/4"
"  -10 / 25  "
"+7"
"0.125"
"1.2e-3"
```

Return a normalized `Fraction`.

Reject:

- empty input;
- division by zero;
- malformed ratios with more than one slash.

In [7]:
def parse_ratio(text: str) -> Fraction:
    if not isinstance(text, str):
        raise TypeError("text must be a string")

    text = text.strip()
    if not text:
        raise ValueError("empty input")

    if text.count("/") > 1:
        raise ValueError("malformed ratio")

    if "/" in text:
        left, right = [part.strip() for part in text.split("/")]
        if not left or not right:
            raise ValueError("missing numerator or denominator")
        return Fraction(int(left), int(right))

    return Fraction(text)


samples = ["3/4", "  -10 / 25  ", "+7", "0.125", "1.2e-3", "", "1/0", "1/2/3"]

for sample in samples:
    try:
        print(f"{sample!r:<14} -> {parse_ratio(sample)!r}")
    except Exception as ex:
        print(f"{sample!r:<14} -> {type(ex).__name__}: {ex}")

'3/4'          -> Fraction(3, 4)
'  -10 / 25  ' -> Fraction(-2, 5)
'+7'           -> Fraction(7, 1)
'0.125'        -> Fraction(1, 8)
'1.2e-3'       -> Fraction(3, 2500)
''             -> ValueError: empty input
'1/0'          -> ZeroDivisionError: Fraction(1, 0)
'1/2/3'        -> ValueError: malformed ratio


### Solution 6

The implementation delegates normalization to `Fraction`.

For slash-based ratios, parsing the numerator and denominator as integers makes the grammar stricter than `Fraction(text)` while still allowing whitespace around `/`.

## Problem 7 — Approximate a Float with a Maximum Denominator

Write `best_fraction_approximation(x, max_denominator)`.

Return a dictionary containing:

- original float;
- exact float fraction;
- limited-denominator approximation;
- absolute error as a `Fraction`;
- absolute error as a float.

In [8]:
def best_fraction_approximation(x: float, max_denominator: int) -> dict[str, object]:
    if not isinstance(x, float):
        raise TypeError("x must be a float")
    if type(max_denominator) is not int or max_denominator < 1:
        raise ValueError("max_denominator must be a positive integer")

    exact = Fraction(x)
    approx = exact.limit_denominator(max_denominator)
    error = abs(exact - approx)

    return {
        "x": x,
        "exact_fraction": exact,
        "approximation": approx,
        "absolute_error": error,
        "absolute_error_float": float(error),
    }


for max_denominator in [10, 100, 500, 10_000]:
    result = best_fraction_approximation(math.pi, max_denominator)
    print(f"max_denominator={max_denominator}")
    for key, value in result.items():
        print(f"  {key}: {value}")
    print()

max_denominator=10
  x: 3.141592653589793
  exact_fraction: 884279719003555/281474976710656
  approximation: 22/7
  absolute_error: 2491454609547/1970324836974592
  absolute_error_float: 0.0012644892673497412

max_denominator=100
  x: 3.141592653589793
  exact_fraction: 884279719003555/281474976710656
  approximation: 311/99
  absolute_error: 4974424337929/27866022694354944
  absolute_error_float: 0.00017851217565170184

max_denominator=500
  x: 3.141592653589793
  exact_fraction: 884279719003555/281474976710656
  approximation: 355/113
  absolute_error: 8484881165/31806672368304128
  absolute_error_float: 2.66764189184887e-07

max_denominator=10000
  x: 3.141592653589793
  exact_fraction: 884279719003555/281474976710656
  approximation: 355/113
  absolute_error: 8484881165/31806672368304128
  absolute_error_float: 2.66764189184887e-07



### Solution 7

`limit_denominator()` finds the closest rational approximation whose denominator does not exceed the specified limit.

Classic approximations to π appear naturally:

- `22/7`
- `311/99`
- `355/113`

## Problem 8 — Rational Weighted Average

Write `weighted_average(values, weights)` using exact rational arithmetic.

Requirements:

- values and weights may be `int`, `str`, or `Fraction`;
- reject empty inputs;
- reject mismatched lengths;
- reject total weight of zero.

In [9]:
def as_fraction(value: int | str | Fraction) -> Fraction:
    if isinstance(value, Fraction):
        return value
    if isinstance(value, int):
        return Fraction(value)
    if isinstance(value, str):
        return Fraction(value)
    raise TypeError(f"cannot convert {value!r} to Fraction safely")


def weighted_average(values, weights) -> Fraction:
    values = list(values)
    weights = list(weights)

    if not values:
        raise ValueError("values must not be empty")
    if len(values) != len(weights):
        raise ValueError("values and weights must have the same length")

    v = [as_fraction(x) for x in values]
    w = [as_fraction(x) for x in weights]

    total_weight = sum(w, start=Fraction(0))
    if total_weight == 0:
        raise ZeroDivisionError("total weight must not be zero")

    return sum((value * weight for value, weight in zip(v, w)), start=Fraction(0)) / total_weight


print(weighted_average([1, 2, 3], [1, 1, 1]))
print(weighted_average(["0.1", "0.2", "0.3"], [1, 2, 1]))
print(weighted_average([Fraction(1, 3), Fraction(2, 3)], ["0.25", "0.75"]))

2
1/5
7/12


### Solution 8

The function converts inputs to `Fraction` before doing arithmetic.

This avoids floating-point rounding errors and gives an exact weighted average.

## Problem 9 — Rational Polynomial Evaluation

Implement `evaluate_polynomial(coefficients, x)` using Horner's method.

The polynomial is represented by coefficients from highest degree to constant term.

Example:

```python
[1, -3, 2]
```

means:

```text
x² - 3x + 2
```

Use exact rational arithmetic.

In [10]:
def evaluate_polynomial(coefficients, x) -> Fraction:
    coefficients = list(coefficients)
    if not coefficients:
        raise ValueError("coefficients must not be empty")

    x = as_fraction(x)
    result = Fraction(0)

    for coefficient in coefficients:
        result = result * x + as_fraction(coefficient)

    return result


coefficients = [1, -3, 2]

for x in [0, 1, 2, Fraction(1, 2), "1.5"]:
    print(f"p({x!r}) = {evaluate_polynomial(coefficients, x)}")

p(0) = 2
p(1) = 0
p(2) = 0
p(Fraction(1, 2)) = 3/4
p('1.5') = -1/4


### Solution 9

Horner's method avoids unnecessary powers and keeps the expression efficient:

```text
((a₀)x + a₁)x + a₂
```

When `x` and the coefficients are rational, every intermediate result remains rational.

## Problem 10 — Exact Probability with Fractions

A biased coin lands heads with probability `3/5`.

Write a function `binomial_probability(n, k, p)` that returns the exact probability of getting exactly `k` heads in `n` flips.

Use:

```python
math.comb(n, k)
```

In [11]:
def binomial_probability(n: int, k: int, p: Fraction) -> Fraction:
    if type(n) is not int or type(k) is not int:
        raise TypeError("n and k must be plain integers")
    if not 0 <= k <= n:
        raise ValueError("k must satisfy 0 <= k <= n")
    if not isinstance(p, Fraction):
        raise TypeError("p must be a Fraction")
    if not 0 <= p <= 1:
        raise ValueError("p must be between 0 and 1")

    return Fraction(math.comb(n, k)) * (p ** k) * ((1 - p) ** (n - k))


p = Fraction(3, 5)

for k in range(6):
    probability = binomial_probability(5, k, p)
    print(f"P(X={k}) = {probability} ≈ {float(probability):.6f}")

print("sum =", sum((binomial_probability(5, k, p) for k in range(6)), start=Fraction(0)))

P(X=0) = 32/3125 ≈ 0.010240
P(X=1) = 48/625 ≈ 0.076800
P(X=2) = 144/625 ≈ 0.230400
P(X=3) = 216/625 ≈ 0.345600
P(X=4) = 162/625 ≈ 0.259200
P(X=5) = 243/3125 ≈ 0.077760
sum = 1


### Solution 10

The binomial probability formula is:

```text
C(n, k) pᵏ (1 - p)ⁿ⁻ᵏ
```

Using `Fraction` keeps the entire calculation exact.

The probabilities sum to exactly `1`.

## Problem 11 — Recover a Repeating Decimal

Write `repeating_decimal_to_fraction(non_repeating, repeating)`.

Examples:

```python
repeating_decimal_to_fraction("0", "3")    -> 1/3
repeating_decimal_to_fraction("1", "6")    -> 1/6
repeating_decimal_to_fraction("12", "34")  -> 611/4950
```

Interpret the input as:

```text
0.non_repeating(repeating repeating repeating ...)
```

For example:

```text
0.12 34 34 34 ...
```

In [12]:
def repeating_decimal_to_fraction(non_repeating: str, repeating: str) -> Fraction:
    if not isinstance(non_repeating, str) or not isinstance(repeating, str):
        raise TypeError("both parts must be strings")

    if non_repeating != "" and not non_repeating.isdigit():
        raise ValueError("non_repeating must be empty or contain digits only")

    if repeating == "" or not repeating.isdigit():
        raise ValueError("repeating part must contain at least one digit")

    a = int(non_repeating) if non_repeating else 0
    b = int(repeating)

    m = len(non_repeating)
    r = len(repeating)

    non_repeating_value = Fraction(a, 10 ** m) if m else Fraction(0)
    repeating_value = Fraction(b, (10 ** m) * (10 ** r - 1))

    return non_repeating_value + repeating_value


examples = [
    ("0", "3"),
    ("1", "6"),
    ("12", "34"),
    ("", "9"),
    ("142857", "142857"),
]

for non_repeating, repeating in examples:
    print(f"0.{non_repeating}({repeating}) = {repeating_decimal_to_fraction(non_repeating, repeating)}")

0.0(3) = 1/30
0.1(6) = 1/6
0.12(34) = 611/4950
0.(9) = 1
0.142857(142857) = 1/7


### Solution 11

A repeating block of length `r` contributes a geometric series.

For `0.12(34)`:

```text
0.12343434... = 12/100 + 34/(100 * 99)
```

`Fraction` automatically reduces the result.

## Problem 12 — Continued Fraction Expansion

Write `continued_fraction(x)` that converts a positive `Fraction` into its finite continued fraction expansion.

Example:

```python
Fraction(355, 113)
```

should produce:

```python
[3, 7, 16]
```

In [13]:
def continued_fraction(x: Fraction) -> list[int]:
    if not isinstance(x, Fraction):
        raise TypeError("x must be a Fraction")
    if x <= 0:
        raise ValueError("x must be positive")

    terms = []
    while x.denominator != 1:
        whole = x.numerator // x.denominator
        terms.append(whole)
        x = 1 / (x - whole)

    terms.append(x.numerator)
    return terms


for x in [Fraction(22, 7), Fraction(355, 113), Fraction(13, 8), Fraction(5, 2)]:
    print(f"{x} -> {continued_fraction(x)}")

22/7 -> [3, 7]
355/113 -> [3, 7, 16]
13/8 -> [1, 1, 1, 1, 2]
5/2 -> [2, 2]


### Solution 12

A continued fraction repeatedly separates the integer part from the reciprocal of the remainder.

For example:

```text
355/113 = 3 + 16/113
113/16 = 7 + 1/16
```

So the expansion is `[3, 7, 16]`.

## Problem 13 — Reconstruct a Fraction from a Continued Fraction

Write `from_continued_fraction(terms)`.

It should invert the previous problem.

Example:

```python
from_continued_fraction([3, 7, 16]) == Fraction(355, 113)
```

In [14]:
def from_continued_fraction(terms) -> Fraction:
    terms = list(terms)

    if not terms:
        raise ValueError("terms must not be empty")
    if any(type(term) is not int for term in terms):
        raise TypeError("all terms must be plain integers")

    result = Fraction(terms[-1])

    for term in reversed(terms[:-1]):
        result = Fraction(term) + Fraction(1, result)

    return result


examples = [
    [3, 7],
    [3, 7, 16],
    [1, 1, 1, 1, 1],
    [2],
]

for terms in examples:
    x = from_continued_fraction(terms)
    print(f"{terms} -> {x} -> back to terms: {continued_fraction(x) if x > 0 else None}")

[3, 7] -> 22/7 -> back to terms: [3, 7]
[3, 7, 16] -> 355/113 -> back to terms: [3, 7, 16]
[1, 1, 1, 1, 1] -> 8/5 -> back to terms: [1, 1, 1, 2]
[2] -> 2 -> back to terms: [2]


### Solution 13

Reconstruction works from right to left.

For `[3, 7, 16]`:

```text
3 + 1 / (7 + 1 / 16)
```

This reconstructs `355/113`.

## Problem 14 — Compare Fractions Without Floating Point

Write `compare_fractions(a, b)` returning:

- `-1` if `a < b`
- `0` if `a == b`
- `1` if `a > b`

Do not convert to `float`.

In [15]:
def compare_fractions(a: Fraction, b: Fraction) -> int:
    if not isinstance(a, Fraction) or not isinstance(b, Fraction):
        raise TypeError("a and b must be Fractions")

    left = a.numerator * b.denominator
    right = b.numerator * a.denominator

    return (left > right) - (left < right)


pairs = [
    (Fraction(1, 3), Fraction(2, 5)),
    (Fraction(2, 4), Fraction(1, 2)),
    (Fraction(355, 113), Fraction(22, 7)),
]

for a, b in pairs:
    result = compare_fractions(a, b)
    symbol = "<" if result < 0 else "==" if result == 0 else ">"
    print(f"{a} {symbol} {b}")

1/3 < 2/5
1/2 == 1/2
355/113 < 22/7


### Solution 14

For positive denominators:

```text
a/b < c/d
```

is equivalent to:

```text
a*d < c*b
```

This avoids precision loss from float conversion.

## Problem 15 — Build a Rational Test Suite

Create tests for the utilities in this notebook.

The tests should cover:

- normalization;
- exact float error;
- parsing;
- weighted averages;
- polynomial evaluation;
- binomial probabilities;
- repeating decimals;
- continued fractions.

In [16]:
def assert_raises(expected_exception: type[BaseException], func, *args, **kwargs) -> None:
    try:
        func(*args, **kwargs)
    except expected_exception:
        return
    except Exception as ex:
        raise AssertionError(
            f"Expected {expected_exception.__name__}, got {type(ex).__name__}: {ex}"
        ) from ex
    else:
        raise AssertionError(f"Expected {expected_exception.__name__}, but no exception was raised")


def run_tests() -> None:
    assert Fraction(8, 16) == Fraction(1, 2)
    assert Fraction(1, -4) == Fraction(-1, 4)

    assert float_error(0.5, Fraction(1, 2)) == 0
    assert float_error(0.3, Fraction(3, 10)) != 0

    assert parse_ratio("  -10 / 25 ") == Fraction(-2, 5)
    assert parse_ratio("0.125") == Fraction(1, 8)
    assert_raises(ValueError, parse_ratio, "")
    assert_raises(ZeroDivisionError, parse_ratio, "1/0")
    assert_raises(ValueError, parse_ratio, "1/2/3")

    assert weighted_average([1, 2, 3], [1, 1, 1]) == 2
    assert weighted_average(["0.1", "0.2"], [1, 1]) == Fraction(3, 20)
    assert_raises(ZeroDivisionError, weighted_average, [1, 2], [1, -1])

    assert evaluate_polynomial([1, -3, 2], 0) == 2
    assert evaluate_polynomial([1, -3, 2], 1) == 0
    assert evaluate_polynomial([1, -3, 2], 2) == 0
    assert evaluate_polynomial([1, -3, 2], Fraction(1, 2)) == Fraction(3, 4)

    assert sum(
        (binomial_probability(5, k, Fraction(3, 5)) for k in range(6)),
        start=Fraction(0),
    ) == 1

    assert repeating_decimal_to_fraction("", "3") == Fraction(1, 3)
    assert repeating_decimal_to_fraction("1", "6") == Fraction(1, 6)
    assert repeating_decimal_to_fraction("12", "34") == Fraction(611, 4950)

    assert continued_fraction(Fraction(355, 113)) == [3, 7, 16]
    assert from_continued_fraction([3, 7, 16]) == Fraction(355, 113)

    assert compare_fractions(Fraction(1, 3), Fraction(2, 5)) == -1
    assert compare_fractions(Fraction(2, 4), Fraction(1, 2)) == 0
    assert compare_fractions(Fraction(355, 113), Fraction(22, 7)) == -1


run_tests()
print("All tests passed.")

All tests passed.


### Solution 15

A strong rational-number test suite checks both success cases and failure cases.

The most important best practices are:

- avoid accidental float conversion;
- test exact equality with `Fraction`;
- test malformed input;
- test mathematical identities such as probability sums and continued-fraction round trips.